# 🏠 Predicción de Precios de Casas con Machine Learning

Este notebook resuelve un problema clásico de **regresión**: predecir el precio de una vivienda
a partir de sus características (ubicación, número de habitaciones, ingreso promedio de la zona, etc.).

**Dataset:** California Housing (incluido en scikit-learn). Es el dataset "estándar" que se usa
hoy en día para practicar este tipo de problemas, ya que reemplazó al antiguo Boston Housing
(retirado de scikit-learn por contener una variable con sesgos éticos).

**Flujo de trabajo:**
1. Cargar y explorar los datos
2. Analizar y visualizar relaciones entre variables
3. Preparar los datos (limpieza, separación en train/test, escalado)
4. Entrenar modelos (Regresión Lineal y Random Forest)
5. Evaluar y comparar resultados
6. Conclusiones

> 💡 Cada línea de código está comentada para que puedas entender exactamente qué hace.


In [ ]:
# En Google Colab, la mayoría de estas librerías ya vienen preinstaladas.
# Esta línea asegura que tengamos las versiones necesarias sin generar errores.
!pip install -q scikit-learn pandas matplotlib seaborn numpy


In [ ]:
# Librería para manejar datos en forma de tablas (DataFrames)
import pandas as pd

# Librería para operaciones numéricas y arreglos
import numpy as np

# Librerías para graficar y visualizar los datos
import matplotlib.pyplot as plt
import seaborn as sns

# Función para cargar el dataset California Housing que viene con scikit-learn
from sklearn.datasets import fetch_california_housing

# Función para dividir los datos en conjunto de entrenamiento y de prueba
from sklearn.model_selection import train_test_split

# Herramienta para escalar (normalizar) las variables numéricas
from sklearn.preprocessing import StandardScaler

# Modelo de Regresión Lineal (modelo simple, sirve como línea base)
from sklearn.linear_model import LinearRegression

# Modelo de Random Forest para regresión (modelo más potente, basado en árboles)
from sklearn.ensemble import RandomForestRegressor

# Métricas para evaluar qué tan bien predicen nuestros modelos
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuramos el estilo de los gráficos para que se vean más prolijos
sns.set_style("whitegrid")

# Fijamos una "semilla" aleatoria para que los resultados sean reproducibles
RANDOM_STATE = 42


## 1. Carga y exploración inicial de los datos

In [ ]:
# Descargamos el dataset California Housing directamente desde scikit-learn
datos_raw = fetch_california_housing(as_frame=True)  # as_frame=True nos devuelve un DataFrame de pandas

# Extraemos el DataFrame completo, que incluye las variables predictoras y el precio objetivo
df = datos_raw.frame

# Renombramos la columna objetivo para que sea más clara (precio medio de la vivienda)
df = df.rename(columns={"MedHouseVal": "PrecioMedioVivienda"})

# Mostramos las primeras 5 filas para tener una primera vista de los datos
df.head()


In [ ]:
# Mostramos información general: tipos de datos, cantidad de filas y columnas no nulas
df.info()


In [ ]:
# Obtenemos estadísticas descriptivas: media, desviación estándar, mínimo, máximo, etc.
df.describe()


In [ ]:
# Contamos cuántos valores nulos (faltantes) hay en cada columna
valores_nulos = df.isnull().sum()

# Imprimimos el resultado para confirmar si es necesario limpiar datos faltantes
print("Valores nulos por columna:")
print(valores_nulos)


## 2. Análisis visual de los datos

In [ ]:
# Creamos una figura de tamaño 8x5 pulgadas
plt.figure(figsize=(8, 5))

# Graficamos un histograma de la variable objetivo (precio de la vivienda)
sns.histplot(df["PrecioMedioVivienda"], bins=50, kde=True, color="steelblue")

# Añadimos título al gráfico
plt.title("Distribución del Precio Medio de Vivienda")

# Etiqueta del eje X
plt.xlabel("Precio medio (en cientos de miles de USD)")

# Etiqueta del eje Y
plt.ylabel("Frecuencia")

# Mostramos el gráfico
plt.show()


In [ ]:
# Calculamos la matriz de correlación entre todas las variables numéricas
matriz_correlacion = df.corr()

# Creamos una figura más grande para que se vea bien el mapa de calor
plt.figure(figsize=(10, 8))

# Graficamos el mapa de calor (heatmap) con los valores de correlación
sns.heatmap(matriz_correlacion, annot=True, fmt=".2f", cmap="coolwarm", square=True)

# Título del gráfico
plt.title("Matriz de Correlación entre Variables")

# Mostramos el gráfico
plt.show()


In [ ]:
# Creamos un gráfico de dispersión para ver cómo se relaciona el ingreso medio
# de la zona (MedInc) con el precio de la vivienda
plt.figure(figsize=(8, 5))

# scatterplot con transparencia (alpha) para ver mejor la densidad de puntos
sns.scatterplot(data=df, x="MedInc", y="PrecioMedioVivienda", alpha=0.3, color="darkorange")

# Título del gráfico
plt.title("Ingreso Medio vs. Precio de Vivienda")

# Etiquetas de los ejes
plt.xlabel("Ingreso medio de la zona (en decenas de miles de USD)")
plt.ylabel("Precio medio de la vivienda")

# Mostramos el gráfico
plt.show()


## 3. Preparación de los datos para el modelo

In [ ]:
# X contiene todas las columnas EXCEPTO la variable objetivo (nuestras variables predictoras)
X = df.drop(columns=["PrecioMedioVivienda"])

# y contiene únicamente la variable que queremos predecir (el precio)
y = df["PrecioMedioVivienda"]

# Mostramos las dimensiones de X e y para confirmar que todo está correcto
print("Forma de X (filas, columnas):", X.shape)
print("Forma de y (filas,):", y.shape)


In [ ]:
# Dividimos los datos: 80% para entrenar el modelo y 20% para probarlo
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                       # Variables predictoras y objetivo
    test_size=0.2,               # 20% de los datos se reservan para prueba
    random_state=RANDOM_STATE    # Semilla fija para que la división sea reproducible
)

# Confirmamos los tamaños de cada conjunto resultante
print("Tamaño de entrenamiento:", X_train.shape[0], "filas")
print("Tamaño de prueba:", X_test.shape[0], "filas")


In [ ]:
# Creamos el escalador que transforma las variables para que tengan media 0 y desviación 1
escalador = StandardScaler()

# Ajustamos el escalador SOLO con los datos de entrenamiento (para evitar fuga de información)
X_train_escalado = escalador.fit_transform(X_train)

# Aplicamos la misma transformación (ya ajustada) al conjunto de prueba
X_test_escalado = escalador.transform(X_test)

# Nota: el escalado es importante para modelos como la Regresión Lineal,
# aunque no es estrictamente necesario para Random Forest.
print("Escalado completado.")


## 4. Entrenamiento de modelos

In [ ]:
# Creamos una instancia del modelo de Regresión Lineal
modelo_lineal = LinearRegression()

# Entrenamos el modelo usando los datos de entrenamiento escalados
modelo_lineal.fit(X_train_escalado, y_train)

# Usamos el modelo entrenado para predecir los precios en el conjunto de prueba
predicciones_lineal = modelo_lineal.predict(X_test_escalado)

print("Modelo de Regresión Lineal entrenado correctamente.")


In [ ]:
# Creamos una instancia del modelo Random Forest con 200 árboles
modelo_rf = RandomForestRegressor(
    n_estimators=200,           # Cantidad de árboles en el bosque
    random_state=RANDOM_STATE,   # Semilla para reproducibilidad
    n_jobs=-1                    # Usa todos los núcleos disponibles del procesador
)

# Entrenamos el modelo con los datos originales (Random Forest no necesita escalado)
modelo_rf.fit(X_train, y_train)

# Generamos las predicciones sobre el conjunto de prueba
predicciones_rf = modelo_rf.predict(X_test)

print("Modelo Random Forest entrenado correctamente.")


## 5. Evaluación y comparación de modelos

In [ ]:
def evaluar_modelo(y_real, y_predicho, nombre_modelo):
    """
    Esta función calcula y muestra las métricas de error para un modelo dado.
    y_real: valores verdaderos del conjunto de prueba
    y_predicho: valores que predijo el modelo
    nombre_modelo: texto para identificar el modelo en el reporte
    """
    # Error absoluto medio: promedio de las diferencias absolutas entre real y predicho
    mae = mean_absolute_error(y_real, y_predicho)

    # Error cuadrático medio: promedio de las diferencias al cuadrado (penaliza más los errores grandes)
    mse = mean_squared_error(y_real, y_predicho)

    # Raíz del error cuadrático medio: en las mismas unidades que la variable objetivo
    rmse = np.sqrt(mse)

    # R^2: qué proporción de la varianza del precio explica el modelo (1.0 = perfecto)
    r2 = r2_score(y_real, y_predicho)

    # Imprimimos un resumen claro de las métricas
    print(f"--- Resultados: {nombre_modelo} ---")
    print(f"MAE  (Error Absoluto Medio):     {mae:.4f}")
    print(f"RMSE (Raíz del Error Cuadrático): {rmse:.4f}")
    print(f"R^2  (Coeficiente de Determinación): {r2:.4f}")
    print()

    # Devolvemos las métricas en un diccionario por si queremos compararlas después
    return {"modelo": nombre_modelo, "MAE": mae, "RMSE": rmse, "R2": r2}


In [ ]:
# Evaluamos el modelo de Regresión Lineal
resultados_lineal = evaluar_modelo(y_test, predicciones_lineal, "Regresión Lineal")

# Evaluamos el modelo Random Forest
resultados_rf = evaluar_modelo(y_test, predicciones_rf, "Random Forest")

# Juntamos ambos resultados en una tabla comparativa
tabla_comparativa = pd.DataFrame([resultados_lineal, resultados_rf])
tabla_comparativa


In [ ]:
# Creamos una figura con dos subgráficos (uno por modelo)
fig, ejes = plt.subplots(1, 2, figsize=(14, 6))

# --- Subgráfico 1: Regresión Lineal ---
ejes[0].scatter(y_test, predicciones_lineal, alpha=0.3, color="steelblue")  # Puntos reales vs predichos
ejes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")  # Línea de predicción perfecta
ejes[0].set_title("Regresión Lineal: Real vs Predicho")
ejes[0].set_xlabel("Precio Real")
ejes[0].set_ylabel("Precio Predicho")

# --- Subgráfico 2: Random Forest ---
ejes[1].scatter(y_test, predicciones_rf, alpha=0.3, color="seagreen")  # Puntos reales vs predichos
ejes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")  # Línea de predicción perfecta
ejes[1].set_title("Random Forest: Real vs Predicho")
ejes[1].set_xlabel("Precio Real")
ejes[1].set_ylabel("Precio Predicho")

# Ajustamos el espaciado entre subgráficos
plt.tight_layout()

# Mostramos la figura completa
plt.show()


In [ ]:
# Extraemos la importancia que el modelo Random Forest le dio a cada variable
importancias = modelo_rf.feature_importances_

# Creamos un DataFrame para ordenar y visualizar mejor las importancias
df_importancias = pd.DataFrame({
    "variable": X.columns,     # Nombres de las variables predictoras
    "importancia": importancias  # Importancia relativa asignada por el modelo
}).sort_values(by="importancia", ascending=False)  # Ordenamos de mayor a menor importancia

# Graficamos la importancia de cada variable en un gráfico de barras horizontal
plt.figure(figsize=(8, 5))
sns.barplot(data=df_importancias, x="importancia", y="variable", color="mediumseagreen")
plt.title("Importancia de Variables (Random Forest)")
plt.xlabel("Importancia relativa")
plt.ylabel("Variable")
plt.show()


## 6. Conclusiones

- El modelo **Random Forest** normalmente obtiene un **R² más alto** y errores (MAE/RMSE) más bajos
  que la **Regresión Lineal**, ya que puede capturar relaciones no lineales entre las variables.
- La variable **`MedInc`** (ingreso medio de la zona) suele ser la más importante para predecir el precio.
- La Regresión Lineal es útil como modelo base (baseline) por su simplicidad e interpretabilidad.
- Próximos pasos posibles: probar otros modelos (XGBoost, Gradient Boosting), ajustar
  hiperparámetros con `GridSearchCV`, o crear nuevas variables (feature engineering).

---
**Autor:** Generado con ayuda de Claude (Anthropic).
